# Adversarial Evasion Attack Demo
## Attack Scenarios and Procedure

This notebook demonstrates a **white-box PGD adversarial evasion attack**.

- **Goal**: craft a perturbation that reduces the reported person count
- **Method**: Projected Gradient Descent (Madry et al., 2018)
- **Constraint**: L-infinity perturbation ≤ ε (visually imperceptible)

> **Screenshot this notebook** for the *Attack Scenarios* and *Outcomes and Consequences* sections.

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from attack import preprocess, pgd_attack, fgsm_attack, postprocess_count

print('Attack module loaded.')

## 1. Load Model and Clean Image

In [ ]:
model = YOLO('../yolov8n.pt')

clean_img = cv2.imread('../outputs/crowd.jpg')
clean_count = postprocess_count(model, clean_img)
print(f'Clean image — people detected: {clean_count}')

## 2. Run PGD Attack

PGD iteratively steps in the direction that **minimises person-class confidence**, projecting back into the ε-ball after each step.

In [ ]:
EPS = 0.08    # max L-inf perturbation (per normalised pixel)
ALPHA = 0.005 # step size per iteration
ITERS = 40    # number of PGD steps

print(f'Running PGD attack (ε={EPS}, α={ALPHA}, iterations={ITERS})...')
adversarial_img = pgd_attack(model, clean_img, eps=EPS, alpha=ALPHA, iterations=ITERS)

adv_count = postprocess_count(model, adversarial_img)
print(f'Adversarial image — people detected: {adv_count}')
print(f'Reduction: {clean_count - adv_count} ({(clean_count - adv_count)/clean_count*100:.1f}%)')

cv2.imwrite('../outputs/crowd_attacked.jpg', adversarial_img)

## 3. Visualise: Clean vs Adversarial

In [ ]:
def annotate(model, bgr_img):
    r = model(bgr_img, conf=0.5, classes=[0], verbose=False)[0]
    return cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB), len(r.boxes)

clean_ann, c_cnt = annotate(model, clean_img)
adv_ann, a_cnt = annotate(model, adversarial_img)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(clean_ann)
axes[0].set_title(f'Clean Image — {c_cnt} people detected', fontsize=13)
axes[0].axis('off')
axes[1].imshow(adv_ann)
axes[1].set_title(f'Adversarial Image — {a_cnt} people detected (PGD ε={EPS})', fontsize=13)
axes[1].axis('off')
plt.suptitle('Clean vs Adversarial Output', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/report_03_attack_comparison.png', dpi=150)
plt.show()
print('Saved: outputs/report_03_attack_comparison.png')

## 4. Perturbation Visualisation

The difference between clean and adversarial image, amplified for visibility:

In [ ]:
diff = cv2.absdiff(clean_img, adversarial_img).astype(np.float32)
diff_amplified = np.clip(diff * 10, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(clean_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(adversarial_img, cv2.COLOR_BGR2RGB))
axes[1].set_title('Adversarial (ε=0.08)')
axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(diff_amplified, cv2.COLOR_BGR2RGB))
axes[2].set_title('Perturbation ×10 (amplified)')
axes[2].axis('off')
plt.suptitle('Adversarial Perturbation — Imperceptible to Human Eye', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/report_04_perturbation.png', dpi=150)
plt.show()
print('Saved: outputs/report_04_perturbation.png')

## 5. Summary Table

In [ ]:
print('=' * 45)
print(f'{"Stage":<30} {"Count":>10}')
print('-' * 45)
print(f'{"Clean image":<30} {clean_count:>10}')
print(f'{"After PGD attack (ε=0.08)":<30} {adv_count:>10}')
print('-' * 45)
print(f'{"Reduction":<30} {clean_count - adv_count:>9} ({(clean_count - adv_count)/clean_count*100:.1f}%)')
print('=' * 45)